# Adaptive BH and adjusted p-values

`select()` applies Benjamini-Hochberg (BH) by default. When anomalies are a sizable share of the batch, BH is conservative: it controls the FDR at `pi0 * alpha`, where `pi0` is the share of true nulls. `method="storey"` estimates `pi0` from the p-values and runs BH at a correspondingly higher level.

This notebook compares the two rules on one labeled batch, reads the same decisions from adjusted p-values, and shows the warning `select()` logs when the calibration set is too small for any discovery.

In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest

from nonconform import ConformalDetector, Split
from nonconform.metrics import false_discovery_rate, statistical_power

## A batch where anomalies are common

30% of the 200 test points are anomalies. The labels `y_family` are used only to evaluate the decisions.

In [ ]:
rng = np.random.default_rng(42)
x_reference = rng.normal(size=(3_000, 4))
x_family = np.vstack([rng.normal(size=(140, 4)), rng.normal(loc=1.75, size=(60, 4))])
y_family = np.r_[np.zeros(140, dtype=int), np.ones(60, dtype=int)]
alpha = 0.1

detector = ConformalDetector(
    detector=IsolationForest(random_state=42),
    strategy=Split(n_calib=1_000),
    seed=42,
).fit(x_reference)

## BH and Storey-BH

Both calls score the same batch against the same 1,000 calibration scores. Choose the rule before looking at the batch; running both and reporting the better one is not covered by either guarantee.

In [ ]:
bh_selected = np.asarray(detector.select(x_family, alpha=alpha))
storey_selected = np.asarray(detector.select(x_family, alpha=alpha, method="storey"))

for name, selected in [("BH", bh_selected), ("Storey-BH", storey_selected)]:
    print(
        f"{name}: {int(selected.sum())} discoveries, "
        f"realized FDP {false_discovery_rate(y_family, selected):.3f}, "
        f"power {statistical_power(y_family, selected):.3f}"
    )

## Adjusted p-values

`last_result` holds the p-values of the latest call. Thresholding its adjusted p-values at `alpha` reproduces both masks without scoring again.

In [ ]:
result = detector.last_result
bh_adjusted = result.adjusted_p_values()
storey_adjusted = result.adjusted_p_values(method="storey")

np.testing.assert_array_equal(bh_adjusted <= alpha, bh_selected)
np.testing.assert_array_equal(storey_adjusted <= alpha, storey_selected)
print(
    "smallest Storey-BH adjusted p-values:", np.round(np.sort(storey_adjusted)[:5], 4)
)

## When the calibration set is too small

Classical conformal p-values cannot fall below `1 / (n + 1)` for `n` calibration scores. A single BH discovery among `m` tests needs a p-value at most `alpha / m`. In the batch below, with `m = 100` and `alpha = 0.1`, that means `n + 1 >= 1000`. With 99 calibration scores, the three far-out points cannot be selected, and `select()` logs a warning on the `nonconform.fdr` logger that states the calibration size needed. The detector with 1,000 calibration scores finds them.

In [ ]:
x_sparse = np.vstack([rng.normal(size=(97, 4)), rng.normal(loc=5.0, size=(3, 4))])

small = ConformalDetector(
    detector=IsolationForest(random_state=42),
    strategy=Split(n_calib=99),
    seed=42,
).fit(x_reference)
small_selected = np.asarray(small.select(x_sparse, alpha=alpha))
large_selected = np.asarray(detector.select(x_sparse, alpha=alpha))
print("discoveries with 99 calibration scores:", int(small_selected.sum()))
print("discoveries with 1,000 calibration scores:", int(large_selected.sum()))

Storey-BH controls the FDR at `alpha` for classical `Empirical` p-values from `Split` or detached calibration with exchangeable, continuous scores. It also accepts `ConditionalEmpirical` with classical ties, and it rejects randomized, KDE-based, weighted, and `DerandomizedSplits` evidence. See the FDR control guide for the full scope.